In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.market_math import precompute_oos_ewma_volatility


# 05 EWMA Volatility Model

Calculate the lagged volatility matrix used by the synthetic options. At date t the variance includes returns only through t minus one.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Load formation and test prices

The same EWMA implementation is used inside the daily backtest.


In [ ]:
train_prices = pd.read_parquet("train_prices.parquet")
test_prices = pd.read_parquet("test_prices.parquet")
display(
    pd.Series(
        {
            "ewma_lambda": cfg.ewma_lambda,
            "formation_sessions": len(train_prices),
            "test_sessions": len(test_prices),
        }
    )
)


## 3. Calculate and check volatility

Save all dates so Module 06 can inspect the actual next-close input for a snapshot signal.


In [ ]:
volatility = precompute_oos_ewma_volatility(train_prices, test_prices, lambda_=cfg.ewma_lambda)
assert np.isfinite(volatility.to_numpy()).all() and (volatility > 0).all().all()
volatility.to_parquet("oos_ewma_volatility.parquet")
display(volatility.head())
volatility.iloc[:, :4].plot(figsize=(10, 4), title="Lagged annualized EWMA volatility")
plt.show()
